In [1]:
import re
import matplotlib.pyplot as plt
import logging
import pickle
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory


In [2]:
# Konfigurasi logging
logging.basicConfig(level=logging.INFO)

logging.info("1. Memuat data...")
columns = ['Harga_Normalized', 'Kamar_Normalized', 'WC_Normalized', 'Parkir_Normalized',
           'Luas_Tanah_Normalized', 'Luas_Bangunan_Normalized', 'Judul_Clean', 'Lokasi_Clean',
           'Deskripsi_Clean', 'Image_Link', 'Property_Link']
df = pd.read_csv('databaru.csv', encoding='utf-8')
df = df[columns]
print(df.head())
print(df.info())


INFO:root:1. Memuat data...


   Harga_Normalized  Kamar_Normalized  WC_Normalized  Parkir_Normalized  \
0          0.664804          0.111111            0.2                0.0   
1          0.469274          0.222222            0.2                1.0   
2          0.449721          0.222222            0.2                0.0   
3          0.208101          0.222222            0.1                0.0   
4          0.092179          0.111111            0.0                0.0   

   Luas_Tanah_Normalized  Luas_Bangunan_Normalized  \
0               0.456522                  0.464115   
1               0.778261                  0.607656   
2               0.378261                  0.464115   
3               0.369565                  0.368421   
4               0.256522                  0.157895   

                                         Judul_Clean         Lokasi_Clean  \
0  rumah baru mewah patra siap huni angsur 79x flatt       ngaglik sleman   
1     jual rumah dekat kampus stie ykpn harga rendah  caturtunggal sle

In [3]:
logging.info("2. Mempersiapkan fitur dan target...")

# Gabungkan teks yang sudah dibersihkan
df['text_combined'] = df['Judul_Clean'] + ' ' + \
                      df['Lokasi_Clean'] + ' ' + \
                      df['Deskripsi_Clean']

# Hitung panjang dokumen
df['doc_length'] = df['text_combined'].str.len()

# Filter dokumen dengan panjang > 3 karakter
df = df[df['text_combined'].str.strip().str.len() > 3]

print("\nJumlah dokumen setelah preprocessing:", len(df))


INFO:root:2. Mempersiapkan fitur dan target...



Jumlah dokumen setelah preprocessing: 7656


In [4]:
logging.info("3. Melakukan TF-IDF Vectorization...")

tfidf = TfidfVectorizer(max_features=1000, min_df=1, stop_words=None)
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
    
try:
    text_features = tfidf.fit_transform(df['text_combined']).toarray()
except ValueError as e:
    print("\n[ERROR] Terjadi error dalam TF-IDF Vectorization:", e)
    print("\n[DEBUG] Contoh isi text_combined (5 dokumen pertama):")
    print(df['text_combined'].head())
    print("\n[DEBUG] Distribusi panjang dokumen:")
    print(df['text_combined'].str.len().describe())
    raise

print("\n[DEBUG] Fitur TF-IDF berhasil dibuat:")
print("Shape:", text_features.shape)
print("Contoh fitur (baris pertama):", text_features[0])


INFO:root:3. Melakukan TF-IDF Vectorization...



[DEBUG] Fitur TF-IDF berhasil dibuat:
Shape: (7656, 1000)
Contoh fitur (baris pertama): [0.         0.         0.         0.         0.         0.
 0.0652176  0.10292151 0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.29371121 0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.09107284 0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0. 

In [5]:
logging.info("4. Mempersiapkan fitur dan target...")

# Fitur teks
features = text_features
target = df['Harga_Normalized'].values

# Hapus sample yang memiliki NaN
mask = ~np.isnan(target)
features = features[mask]
target = target[mask]


INFO:root:4. Mempersiapkan fitur dan target...


In [6]:
logging.info("5. Membagi data latih dan uji...")

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)


INFO:root:5. Membagi data latih dan uji...


In [7]:
from sklearn.preprocessing import StandardScaler

# Kolom numerik yang relevan
numeric_columns = [
    'Harga_Normalized', 'Kamar_Normalized', 'WC_Normalized',
    'Parkir_Normalized', 'Luas_Tanah_Normalized', 'Luas_Bangunan_Normalized'
]

# Ambil fitur numerik dari dataframe
numeric_features = df[numeric_columns].values

# Normalisasi fitur numerik
scaler = StandardScaler()
numeric_features_scaled = scaler.fit_transform(numeric_features)

# Menambahkan fitur turunan yang relevan
numeric_features_final = np.column_stack([
    numeric_features_scaled,
    df['Harga_Normalized'] / df['Luas_Bangunan_Normalized'],
    df['Luas_Bangunan_Normalized'] / df['Luas_Tanah_Normalized'],
    df['Kamar_Normalized'] * df['WC_Normalized']
])

print("\n[DEBUG] Fitur numerik setelah normalisasi dan penambahan fitur:")
print("Shape:", numeric_features_final.shape)
print("Contoh fitur (baris pertama):", numeric_features_final[0])



[DEBUG] Fitur numerik setelah normalisasi dan penambahan fitur:
Shape: (7656, 9)
Contoh fitur (baris pertama): [ 1.8122785  -0.95270756  1.07054514 -0.58097006  0.04370024  0.35094668
  1.43241375  1.01663249  0.02222222]


In [8]:
logging.info("5. Membagi data latih dan uji...")

X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    text_features, numeric_features_final, target, test_size=0.2, random_state=42
)
X_text_train, X_text_val, X_num_train, X_num_val, y_train, y_val = train_test_split(
    X_text_train, X_num_train, y_train, test_size=0.2, random_state=42
)


INFO:root:5. Membagi data latih dan uji...


In [22]:
logging.info("6. Membuat dan melatih model teks...")

def r2_keras(y_true, y_pred):
    SS_res =  tf.reduce_sum(tf.square(y_true - y_pred)) 
    SS_tot = tf.reduce_sum(tf.square(y_true - tf.reduce_mean(y_true))) 
    return (1 - SS_res/(SS_tot + tf.keras.backend.epsilon()))

text_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_text_train.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1)
])

text_model.compile(
    optimizer=Adam(learning_rate=0.001), 
    loss='mean_squared_error',
    metrics=['mae', r2_keras]
)

early_stopping = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

text_history = text_model.fit(
    X_text_train, y_train, 
    validation_data=(X_text_val, y_val),
    epochs=100, 
    batch_size=32, 
    callbacks=[early_stopping],
    verbose=1
)

# Print final metrics
y_pred = text_model.predict(X_text_test)
print("\nFinal Test Metrics:")
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"MSE: {mean_squared_error(y_test, y_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")


INFO:root:6. Membuat dan melatih model teks...


Epoch 1/100
154/154 [==============================] - 2s 4ms/step - loss: 0.0320 - mae: 0.1266 - r2_keras: 0.1905 - val_loss: 0.0137 - val_mae: 0.0809 - val_r2_keras: 0.6566
Epoch 2/100
154/154 [==============================] - 1s 4ms/step - loss: 0.0146 - mae: 0.0848 - r2_keras: 0.6197 - val_loss: 0.0115 - val_mae: 0.0710 - val_r2_keras: 0.7085
Epoch 3/100
154/154 [==============================] - 1s 4ms/step - loss: 0.0111 - mae: 0.0740 - r2_keras: 0.7006 - val_loss: 0.0109 - val_mae: 0.0683 - val_r2_keras: 0.7181
Epoch 4/100
154/154 [==============================] - 1s 4ms/step - loss: 0.0091 - mae: 0.0677 - r2_keras: 0.7587 - val_loss: 0.0095 - val_mae: 0.0641 - val_r2_keras: 0.7554
Epoch 5/100
154/154 [==============================] - 1s 3ms/step - loss: 0.0073 - mae: 0.0605 - r2_keras: 0.8050 - val_loss: 0.0110 - val_mae: 0.0701 - val_r2_keras: 0.7148
Epoch 6/100
154/154 [==============================] - 0s 3ms/step - loss: 0.0064 - mae: 0.0563 - r2_keras: 0.8334 - val_loss

In [16]:
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import BatchNormalization

logging.info("7. Membuat dan melatih model numerik...")

numeric_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_num_train.shape[1],), kernel_regularizer=l2(0.01)),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
    BatchNormalization(),
    Dropout(0.2),
    Dense(32, activation='relu', kernel_regularizer=l2(0.01)),
    BatchNormalization(),
    Dense(16, activation='relu'),
    Dense(1)
])

numeric_model.compile(
    optimizer=Adam(learning_rate=0.0005), 
    loss='mean_squared_error',
    metrics=['mae', r2_keras]
)

# Add learning rate reduction callback
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=5,
    min_lr=0.00001
)

numeric_history = numeric_model.fit(
    X_num_train, y_train, 
    validation_data=(X_num_val, y_val),
    epochs=100, 
    batch_size=32, 
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

# Print final metrics
y_pred = numeric_model.predict(X_num_test)
print("\nFinal Test Metrics:")
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"MSE: {mean_squared_error(y_test, y_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")


INFO:root:7. Membuat dan melatih model numerik...


Epoch 1/100
154/154 [==============================] - 3s 5ms/step - loss: 1.7170 - mae: 0.4250 - r2_keras: -7.9309 - val_loss: 1.3983 - val_mae: 0.2197 - val_r2_keras: -0.7841 - lr: 5.0000e-04
Epoch 2/100
154/154 [==============================] - 1s 3ms/step - loss: 1.3549 - mae: 0.2318 - r2_keras: -1.4462 - val_loss: 1.2329 - val_mae: 0.1613 - val_r2_keras: -0.1465 - lr: 5.0000e-04
Epoch 3/100
154/154 [==============================] - 1s 4ms/step - loss: 1.1644 - mae: 0.1724 - r2_keras: -0.3601 - val_loss: 1.0508 - val_mae: 0.0933 - val_r2_keras: 0.5805 - lr: 5.0000e-04
Epoch 4/100
154/154 [==============================] - 1s 3ms/step - loss: 0.9919 - mae: 0.1402 - r2_keras: 0.0905 - val_loss: 0.8915 - val_mae: 0.0747 - val_r2_keras: 0.7287 - lr: 5.0000e-04
Epoch 5/100
154/154 [==============================] - 1s 4ms/step - loss: 0.8328 - mae: 0.1194 - r2_keras: 0.3472 - val_loss: 0.7434 - val_mae: 0.0660 - val_r2_keras: 0.8017 - lr: 5.0000e-04
Epoch 6/100
154/154 [==============

In [17]:
logging.info("8. Mengevaluasi model...")

# Visualisasi Training Text Model
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.title('Text Model Training Performance')
plt.plot(text_history.history['loss'], label='Training Loss')
plt.plot(text_history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()

plt.subplot(1, 2, 2)
plt.title('Text Model Training MAE')
plt.plot(text_history.history['mae'], label='Training MAE')
plt.plot(text_history.history['val_mae'], label='Validation MAE')
plt.xlabel('Epochs')
plt.ylabel('Mean Absolute Error')
plt.legend()

plt.tight_layout()
plt.savefig('text_model_training_performance.png')
plt.close()

# Visualisasi Training Numeric Model
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.title('Numeric Model Training Performance')
plt.plot(numeric_history.history['loss'], label='Training Loss')
plt.plot(numeric_history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()

plt.subplot(1, 2, 2)
plt.title('Numeric Model Training MAE')
plt.plot(numeric_history.history['mae'], label='Training MAE')
plt.plot(numeric_history.history['val_mae'], label='Validation MAE')
plt.xlabel('Epochs')
plt.ylabel('Mean Absolute Error')
plt.legend()

plt.tight_layout()
plt.savefig('numeric_model_training_performance.png')
plt.close()

# Scatter Plots untuk Actual vs Predicted Values
# Prediksi untuk model teks
y_pred_text = text_model.predict(X_text_test).flatten()
y_pred_numeric = numeric_model.predict(X_num_test).flatten()

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.title('Text Model: Actual vs Predicted')
plt.scatter(y_test, y_pred_text, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')

plt.subplot(1, 2, 2)
plt.title('Numeric Model: Actual vs Predicted')
plt.scatter(y_test, y_pred_numeric, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')

plt.tight_layout()
plt.savefig('model_prediction_comparison.png')
plt.close()

print("Visualization images have been saved:")
print("1. text_model_training_performance.png")
print("2. numeric_model_training_performance.png")
print("3. model_prediction_comparison.png")


INFO:root:8. Mengevaluasi model...


48/48 [==============================] - 0s 2ms/step
Visualization images have been saved:
1. text_model_training_performance.png
2. numeric_model_training_performance.png
3. model_prediction_comparison.png


In [40]:
logging.info("9. Menyimpan model...")

text_model.save('text_model.keras')
numeric_model.save('numeric_model.keras')

print("Models have been saved as text_model.keras and numeric_model.keras")


INFO:root:9. Menyimpan model...


Models have been saved as text_model.keras and numeric_model.keras


In [41]:
logging.info("9. Menyimpan model...")

text_model.save('text_model.h5')
numeric_model.save('numeric_model.h5')

print("Models have been saved as text_model.h5 and numeric_model.h5")


INFO:root:9. Menyimpan model...


Models have been saved as text_model.h5 and numeric_model.h5


c:\Users\L E N O V O\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
